# Lecture: Building a Reproducible Analysis Workflow

In this lecture, we will organize an analysis around a question. As each step creates a need, we will introduce the Python and Pandas tools required to continue.


## Learning goals

By the end of this lecture, you should be able to:

- organize an analysis using a reproducible workflow;
- identify the observations and features needed to answer a question;
- convert text to Pandas date-time values;
- use date-time methods to create a new feature;
- explain scalar broadcasting and vectorized operations;
- group observations and calculate a summary for each group;
- use a numeric indicator to define and summarize groups;
- check whether an analytical result is reasonable; and
- distinguish evidence, conclusions, and limitations.


## The reproducible analysis workflow

We will use seven steps:

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Operation:** What should the code select, calculate, or compare?
4. **Check:** Does the result have the expected observations, values, and units?
5. **Evidence:** Which result directly answers the question?
6. **Conclusion:** What claim is supported?
7. **Limitation:** What should we avoid concluding?


Code that runs without an error can still answer the wrong question. A reproducible analysis preserves the code and records how each result supports the conclusion.


## Step 1: Question

This lecture uses a snapshot of worldwide seismic events recorded by the U.S. Geological Survey from January through June 2026. The file includes events with a reported magnitude of at least 2.5.

### Which month had the highest percentage of recorded magnitude-5.0+ events?


### Discussion: predict the month

1. Which month do you predict will have the greatest proportion of recorded magnitude-5.0+ events?
2. Why should we compare percentages rather than raw counts of magnitude-5.0+ events?


### Discussion notes

1. Predictions will vary. Recording a prediction makes our expectations visible before we calculate the result.
2. The total number of recorded events may differ from month to month. A raw count of magnitude-5.0+ events does not account for those different monthly totals. A percentage compares the larger-event count with the total recorded count for the same month.


## Step 2: Data

Each observation represents one recorded seismic event. We will need:

- `event_time` to identify the month;
- `magnitude` to identify magnitude-5.0+ events; and
- `event_id` to count the recorded events.


### Data dictionary

Before writing the analysis, review all the features available in the file.

| Feature | Feature type | Description |
| --- | --- | --- |
| `event_id` | Identifier | Distinguishes one event record from another |
| `event_time` | Date-time | Date and time when the event began, recorded in UTC |
| `magnitude` | Quantitative | Estimate of the event's size |
| `depth_km` | Quantitative | Estimated depth where rupture began, in kilometers |
| `magnitude_type` | Categorical | Method used to calculate magnitude |
| `reporting_network` | Categorical | Network that supplied the preferred event information |
| `event_type` | Categorical | Classification such as earthquake or mining explosion |
| `review_status` | Categorical | Whether the record was reviewed or remained automatic |
| `place` | Text | Description of the event's location |
| `horizontal_error_km` | Quantitative | Estimate of horizontal location uncertainty, in kilometers |


### Load and inspect the data


In [ ]:
import pandas as pd

earthquakes = pd.read_csv("data/usgs_earthquakes_2026_h1_m25.csv")


In [ ]:
earthquakes.head()


In [ ]:
earthquakes.tail()


In [ ]:
earthquakes.info()


In [ ]:
earthquakes.isnull().sum()


### Discussion: check the data

1. What does each row represent?
2. Does the file contain the features needed to answer the question?
3. How is `event_time` currently stored?
4. Are either `event_time`, `magnitude`, or `event_id` missing any values?


### Discussion notes

1. Each row represents one recorded seismic event.
2. Yes. The file contains `event_time`, `magnitude`, and `event_id`.
3. `event_time` is stored as text rather than as a Pandas date-time value.
4. None of the three features needed for this analysis contain missing values.


## Step 3: Operation

To answer the question, we need to:

1. convert `event_time` from text to date-time values;
2. identify the month of each event;
3. identify which events had magnitudes of at least 5.0;
4. group the observations by month;
5. count all events and magnitude-5.0+ events in each month; and
6. calculate a monthly percentage.


### Create the event-month feature

`pd.to_datetime()` converts the text values to date-time values, and `utc=True` retains the UTC time-zone information indicated by the timestamps. The `.dt` accessor then provides date-time operations for the converted Series. Here, `.dt.month_name()` extracts the name of the month from every timestamp.

The complete operation is assigned directly to a new column in the `earthquakes` DataFrame. We do not need to save the intermediate date-time Series separately when it will not be reused.


In [ ]:
earthquakes["event_month"] = (
    pd.to_datetime(earthquakes["event_time"], utc=True)
    .dt.month_name()
)

earthquakes[["event_time", "event_month"]].head()


### Identify magnitude-5.0+ events

The comparison below asks whether each value in the `magnitude` Series is at least 5.0. The comparison initially produces `True` and `False`; `.astype(int)` converts those values to 1 and 0 before the result is assigned directly to a new column.


In [ ]:
earthquakes["magnitude_5_plus"] = (
    (earthquakes["magnitude"] >= 5.0)
    .astype(int)
)

earthquakes[["magnitude", "magnitude_5_plus"]].head()


Pandas **broadcasts** the single value `5.0` across the entire Series and performs the comparison for every observation. This is a **vectorized operation**: we operate on the Series without writing a loop for its individual values. In the new indicator column, 1 identifies a magnitude-5.0+ event and 0 identifies an event below 5.0.


### Group and summarize the observations

`.groupby("event_month")` forms one group for each month. `.agg()` then calculates two summaries for every group:

- `count` counts all event identifiers; and
- `sum` counts the 1 values in the numeric `magnitude_5_plus` indicator.


In [ ]:
monthly_summary = (
    earthquakes
    .groupby("event_month")
    .agg(
        event_count=("event_id", "count"),
        magnitude_5_plus_count=("magnitude_5_plus", "sum")
    )
)

monthly_summary


### Calculate the percentage

Divide each month's magnitude-5.0+ count by its total event count and multiply by 100. These arithmetic operations are vectorized across the two columns.


In [ ]:
monthly_summary["magnitude_5_plus_percent"] = (
    monthly_summary["magnitude_5_plus_count"]
    / monthly_summary["event_count"]
    * 100
)

monthly_summary


Sort the months from the greatest percentage to the smallest so the result that answers the question appears first.


In [ ]:
monthly_summary = monthly_summary.sort_values(
    by="magnitude_5_plus_percent",
    ascending=False
)

monthly_summary


## Step 4: Check

A sensible result should satisfy all of the following conditions:

- the table contains six months;
- the monthly event counts add to the number of observations in the original DataFrame;
- the monthly magnitude-5.0+ counts add to the total number of 1 values in the indicator column; and
- every percentage falls between 0 and 100.


In [ ]:
print("Number of months:", len(monthly_summary))
print("Monthly counts:", monthly_summary["event_count"].sum())
print("Original observations:", len(earthquakes))
print(
    "Monthly magnitude-5.0+ count:",
    monthly_summary["magnitude_5_plus_count"].sum()
)
print(
    "Total magnitude-5.0+ indicators:",
    earthquakes["magnitude_5_plus"].sum()
)
print(
    "Percentages are between 0 and 100:",
    monthly_summary["magnitude_5_plus_percent"].between(0, 100).all()
)


### Discussion: check the result

Did the result pass all four checks?


**Discussion notes:** Yes. The table has six rows, both sets of counts match their expected totals, and every percentage is between 0 and 100.


## Step 5: Evidence

Display the counts and percentage needed to answer the question.


In [ ]:
monthly_summary[[
    "event_count",
    "magnitude_5_plus_count",
    "magnitude_5_plus_percent"
]]


### Discussion: identify the evidence

Which row provides the evidence needed to answer the question?


**Discussion notes:** Because the table is sorted from the greatest percentage to the smallest, the first row provides the evidence needed to answer the question.


## Step 6: Conclusion

### Discussion: form a conclusion

Which month had the highest percentage of recorded magnitude-5.0+ events, and what was that percentage?


**Discussion notes:** In this snapshot, January had the highest percentage of recorded magnitude-5.0+ events. Approximately 7.8% of its recorded magnitude-2.5+ events had magnitudes of at least 5.0.


## Step 7: Limitation

### Discussion: identify a limitation

Does this result establish that earthquakes are generally stronger in January?


**Discussion notes:** No. The result describes recorded magnitude-2.5+ events from one six-month period. It does not establish a seasonal pattern or show that all earthquakes are generally stronger in January.


## A second workflow example

We can apply the workflow to a different question without repeating the general inspection of the DataFrame.


## Step 1: Question

### On average in our dataset, which events had greater magnitudes: reviewed events or automatically recorded events?


### Discussion: predict the comparison

Which group do you predict will have the greater average magnitude? Why might the USGS review some event records but leave others automatic?


**Discussion notes:** Predictions will vary. One reasonable prediction is that reviewed records will have a greater average magnitude because larger or more consequential events may be more likely to receive human attention. The comparison can describe an association, but it cannot establish why a record was reviewed.


## Step 2: Data

We need `review_status` to form the groups, `magnitude` to calculate the group means, and `event_id` to count the observations in each group.


## Step 3: Operation

Group the observations by `review_status`. For each group, calculate the number of observations and the mean magnitude.


In [ ]:
review_summary = (
    earthquakes
    .groupby("review_status")
    .agg(
        event_count=("event_id", "count"),
        mean_magnitude=("magnitude", "mean")
    )
)

review_summary


## Step 4: Check

Check that the two group counts add to the number of observations in `earthquakes` and that the magnitude values needed for the comparison are complete.


In [ ]:
print("Original observations:", len(earthquakes))
print("Sum of group counts:", review_summary["event_count"].sum())
print(
    "Missing magnitude values:",
    earthquakes["magnitude"].isnull().sum()
)


### Discussion: check the result

Did the result pass the checks? How do the group sizes compare?


**Discussion notes:** Yes. The two group counts add to all 14,138 observations, and no magnitude values are missing. The groups are very unequal: 14,061 records were reviewed, while only 77 remained automatic.


## Step 5: Evidence

Display the group counts and mean magnitudes rounded to two decimal places.


In [ ]:
review_summary.round(2)


## Step 6: Conclusion

### Discussion: form a conclusion

Which review-status group had the greater average magnitude, and what were the two means?


**Discussion notes:** Reviewed events had the greater average magnitude. Reviewed records averaged approximately 3.82, compared with approximately 2.78 for automatic records.


## Step 7: Limitation

### Discussion: identify a limitation

Does this comparison establish that human review changes an event's magnitude?


**Discussion notes:** No. Review status was not assigned randomly, and the two groups are very unequal. Larger or more consequential events may be more likely to receive review. The result describes an association between review status and recorded magnitude; it does not show that review caused an event to have a greater magnitude.


## A third workflow example

The numeric magnitude indicator created for the first question can also define two groups for another comparison.


## Step 1: Question

### Were magnitude-5.0+ events deeper or shallower on average than magnitude-2.5–4.9 events?


### Discussion: predict the comparison

Which magnitude group do you predict will be deeper on average?


**Discussion notes:** Predictions will vary. The purpose of recording a prediction is to make our expectations visible before we calculate the result.


## Step 2: Data

We need the numeric `magnitude_5_plus` indicator to form the groups, `depth_km` to calculate the group means, and `event_id` to count the observations. In the indicator column, 0 represents a magnitude from 2.5 through 4.9 and 1 represents a magnitude of at least 5.0.


## Step 3: Operation

Group the observations by `magnitude_5_plus`. For each group, calculate the number of observations and the mean depth.


In [ ]:
depth_by_magnitude = (
    earthquakes
    .groupby("magnitude_5_plus")
    .agg(
        event_count=("event_id", "count"),
        mean_depth_km=("depth_km", "mean")
    )
)

depth_by_magnitude


## Step 4: Check

Check that the group counts add to the number of observations in `earthquakes` and that the depth values needed for the comparison are complete.


In [ ]:
print("Original observations:", len(earthquakes))
print(
    "Sum of group counts:",
    depth_by_magnitude["event_count"].sum()
)
print(
    "Missing depth values:",
    earthquakes["depth_km"].isnull().sum()
)


### Discussion: check the result

Did the result pass the checks? How do the group sizes compare?


**Discussion notes:** Yes. The two group counts add to all 14,138 observations, and no depth values are missing. The groups are unequal: 13,220 events had magnitudes from 2.5 through 4.9, while 918 events had magnitudes of at least 5.0.


## Step 5: Evidence

Display the group counts and mean depths rounded to two decimal places.


In [ ]:
depth_by_magnitude.round(2)


## Step 6: Conclusion

### Discussion: form a conclusion

Which magnitude group was deeper on average, and what were the two mean depths?


**Discussion notes:** Magnitude-2.5–4.9 events were deeper on average. Their mean recorded depth was approximately 58.75 kilometers, compared with approximately 50.88 kilometers for magnitude-5.0+ events.


## Step 7: Limitation

### Discussion: identify a limitation

Why should the lower-magnitude group not be interpreted as representing all smaller earthquakes?


**Discussion notes:** The source file includes only events with reported magnitudes of at least 2.5. Events below magnitude 2.5 are absent, so the 0 group represents only recorded magnitude-2.5–4.9 events rather than all smaller earthquakes. The comparison also describes an association and does not show that magnitude determines depth.


## Discussion: review

1. Why did the analysis compare percentages instead of only monthly counts?
2. What did scalar broadcasting accomplish when we compared `magnitude` with `5.0`?
3. What did `.astype(int)` do to the results of the magnitude comparison?
4. Why did we check the grouped counts against the original data?
5. Why should the reviewed and automatic magnitude means be interpreted alongside their group counts?
6. In the depth comparison, what do the indicator values 0 and 1 represent?
7. Why does the depth comparison not describe earthquakes below magnitude 2.5?


### Discussion notes

1. The monthly totals differed, so percentages compared the larger-event count with the total for the same month.
2. Broadcasting applied the comparison with `5.0` to every magnitude in the Series.
3. `.astype(int)` converted `True` to 1 and `False` to 0, producing a numeric indicator.
4. Matching totals helped confirm that grouping did not omit or duplicate observations.
5. The means are based on 14,061 reviewed records but only 77 automatic records. The smaller group provides much less evidence and may represent a systematically different set of events.
6. A value of 0 represents a magnitude from 2.5 through 4.9, while 1 represents a magnitude of at least 5.0.
7. The source query excluded events below magnitude 2.5, so those smaller events are not represented in either group.


## Summary

We used the analysis workflow to answer three related questions without repeating unnecessary inspection. Along the way, we converted date-time values, extracted months, created a numeric indicator with scalar broadcasting and a vectorized comparison, grouped observations, calculated counts, percentages, and means, checked results, identified evidence, formed conclusions, and stated limitations.
